In [1]:
import collections
import matplotlib.pyplot as plt
from IPython import display
#import itertools as itr
import numpy as np
import math
#from sklearn.metrics import mean_squared_error
#from random import randrange
#from tabulate import tabulate

import torch
from torch import Tensor, nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
#from torch.func import functional_call, vmap, vjp, jvp, grad

#from scipy.linalg import svdvals, norm
#from scipy.special import softmax
#from scipy.optimize import minimize_scalar

from common.const import DATASET_PATH
from common.util import *
from common.meta import MetaData
from common.optimise import  reduce_to_active
from common.optimise_v2 import OptimiserEtaSoftmaxArmihoNorm2Base, OptimiserEtaSoftmaxArmihoNorm1Base

from torchvision.datasets import CIFAR10
from torchvision import transforms

import logging

d:\Projects-my\ml\Ml-readinggroup\Roberts_Yaida\chapterInf\common\optimise_v2.py:76: SyntaxWarning: invalid escape sequence '\i'
  '''


In [2]:
logging.basicConfig(filename="log_convcrity_cifar10_epochsdV2.log",
                level=logging.INFO,
                format="%(levelname)s: %(asctime)s %(message)s")
#                datefmt="%m/%d/%Y %I:%M:%S")


#### CIFAR10 dataset

In [3]:
# Convert from PIL to torch.Tensort
# and normalize each pixel from [0, 255] range to [0.0, 1.0]
base_transforms = transforms.ToTensor()

# An augmentation that randomly (with a probability equal to 0.5)
# flips the image horizontally
# This will prevent overfitting and make the model more robust
aug_transforms = transforms.RandomHorizontalFlip(p=0.5)

# Gather all transforms together
train_transforms = transforms.Compose([
    base_transforms,
    aug_transforms
])

train_dataset = CIFAR10(root=DATASET_PATH, train=True, download=True, transform=train_transforms)

# Note that we only use `base_transforms` for test dataset
test_dataset = CIFAR10(root=DATASET_PATH, train=False, download=True, transform=base_transforms)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
BATCH_SIZE = 64 #64 #32

# `pin_memory` speed up processing if you use GPU
# `num_workers` also speed up processing since use additional process
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                             num_workers=0, pin_memory=True)

In [5]:
#Weights distribution variances are set as in (5.67)
slope_plus, slope_minus=1.0, 0.1
cb, cw = 0, 2.0/(slope_plus**2.0 + slope_minus**2.0)

INPUT_DIM=32*32 #*3 USING 0.299 ∙ Red + 0.587 ∙ Green + 0.114 ∙ Blue for gray-transform
OUTPUT_DIM = 10 #26  # num of classes
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

lb, lw = 0.005, 5.0
meta = MetaData(input_dim = INPUT_DIM, output_dim = OUTPUT_DIM, batch_size = BATCH_SIZE, device=DEVICE, reduction='mean') #'mean'



##### UTILS

#### Compare optimisers

In [6]:
def init_weights_copy(net, params):
    net.init_linear_zeros(net.conv1, True)
    net.init_linear_zeros(net.conv2, True)
    net.init_linear_zeros(net.conv3, True)
    net.init_linear_zeros(net.conv4, True)
    net.init_linear_zeros(net.linear5, True)
    net.init_linear_zeros(net.linear6, True)
    with torch.no_grad():
        net.conv1.weight += torch.from_numpy(params['0.weight']).to(DEVICE)
        net.conv1.bias += torch.from_numpy(params['0.bias']).to(DEVICE)
        net.conv2.weight += torch.from_numpy(params['2.weight']).to(DEVICE)
        net.conv2.bias += torch.from_numpy(params['2.bias']).to(DEVICE)
        net.conv3.weight += torch.from_numpy(params['6.weight']).to(DEVICE)
        net.conv3.bias += torch.from_numpy(params['6.bias']).to(DEVICE)
        net.conv4.weight += torch.from_numpy(params['8.weight']).to(DEVICE)
        net.conv4.bias += torch.from_numpy(params['8.bias']).to(DEVICE)
        net.linear5.weight += torch.from_numpy(params['13.weight']).to(DEVICE)
        net.linear5.bias += torch.from_numpy(params['13.bias']).to(DEVICE)
        net.linear6.weight += torch.from_numpy(params['16.weight']).to(DEVICE)
        net.linear6.bias += torch.from_numpy(params['16.bias']).to(DEVICE)

def print_comparison(HISTORY_NORM1, HISTORY_NORM2, HISTORY_OPTFIX):
    #start_epoch, finish_epoch = 10, 20
    display.clear_output()
    fig, axes = plt.subplots(2, 1, figsize=(12, 16))
    axes[0].set_title('Loss (Cross Entropy)')
    #axes[0].set_xlim(start_epoch-1, finish_epoch)
    #axes[0].plot(HISTORY_NORM1['train_loss'][2:40], color='g', ls='dotted', alpha=.5, label='Train norm1')
    #axes[0].plot(HISTORY_NORM2['train_loss'][2:40], color='b', ls='dotted', alpha=.5, label='Train norm2')
    #axes[0].plot(HISTORY_ZHS['train_loss'][2:40], color='r', ls='dotted', alpha=.5, label='Train Zhang-simple')
    #axes[0].plot(HISTORY_ZHL['train_loss'][2:40], color='orange', ls='dotted', alpha=.5, label='Train Zhang-lambda')
    #axes[0].plot(HISTORY_OPTADA['train_loss'][2:40], color='brown', ls='dotted', alpha=.5, label='Train direct Zhang-lambda')
    #axes[0].plot(HISTORY_NORM1['test_loss'], color='g', alpha=.5, label='Test norm1')
    axes[0].plot(HISTORY_NORM2['test_loss'], color='b', alpha=.5, label='Test norm2')
    #axes[0].plot(HISTORY_ZHS['test_loss'], color='r', alpha=.5, label='Test Zhang-simple')
    #axes[0].plot(HISTORY_ZHL['test_loss'], color='orange', alpha=.5, label='Test Zhang-lambda')
    #axes[0].plot(HISTORY_OPTADA['test_loss'], color='purple', alpha=.5, label='Test Zhang-simple optimiser')
    axes[0].plot(HISTORY_OPTFIX['test_loss'], color='grey', alpha=.5, label='Test Nesterov-fix optimiser')
    axes[0].grid()
    axes[0].legend()
    axes[0].set_xlabel("Epoch (series of gradient descent steps)")
    axes[0].set_ylabel("Loss value")

    axes[1].set_title('Accuracy')
    #axes[1].set_xlim(10, 20)
    #axes[1].plot(HISTORY_NORM1['train_accuracy'][2:40], color='g', alpha=.5, ls='dotted', label='Train norm1')
    #axes[1].plot(HISTORY_NORM2['train_accuracy'][2:40], color='b', alpha=.5, ls='dotted', label='Train norm2')
    #axes[1].plot(HISTORY_ZHS['train_accuracy'][2:40], color='r', alpha=.5, ls='dotted', label='Train Zhang-simple')
    #axes[1].plot(HISTORY_ZHL['train_accuracy'][2:40], color='orange', alpha=.5, ls='dotted', label='Train Zhang-lambda')
    #axes[1].plot(HISTORY_OPTADA['train_accuracy'][2:40], color='brown', alpha=.5, ls='dotted', label='Train direct Zhang-lambda')
    #axes[1].plot(HISTORY_NORM1['test_accuracy'], color='g', alpha=.5, label='Test norm1')
    axes[1].plot(HISTORY_NORM2['test_accuracy'], color='b', alpha=.5, label='Test norm2')
    #axes[1].plot(HISTORY_ZHS['test_accuracy'], color='r', alpha=.5, label='Test Zhang-simple')
    #axes[1].plot(HISTORY_ZHL['test_accuracy'], color='orange', alpha=.5, label='Test Zhang-lambda')
    #axes[1].plot(HISTORY_OPTADA['test_accuracy'], color='purple', alpha=.5, label='Test Zhang-simple optimiser')
    axes[1].plot(HISTORY_OPTFIX['test_accuracy'], color='grey', alpha=.5, label='Test Nesterov-fix optimiser')
    axes[1].grid()
    axes[1].legend()
    axes[1].set_xlabel("Epoch (series of gradient descent steps)")
    axes[1].set_ylabel("Accuracy value")

    #fig.tight_layout()
    #fig.subplots_adjust(top=0.95)
    dummy=fig.suptitle("Comparison for the same FFN-structure for different optimisers")

    plt.show()

#### Reference CNN

In [7]:
def make_model():
    
    model = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(3, 3), padding='same'),
        nn.LeakyReLU(0.1),
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding='same'),
        nn.LeakyReLU(0.1),
        nn.MaxPool2d(kernel_size = (2,2)),
        nn.Dropout(0.25),
        nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3, 3), padding='same'),
        nn.LeakyReLU(0.1),
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), padding='same'),
        nn.LeakyReLU(0.1),
        nn.MaxPool2d(kernel_size = (2,2)),
        nn.Dropout(0.25),
        nn.Flatten(),
        nn.Linear(4096, 256),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.5),
        nn.Linear(256, 10)        
    )

    return model

#### Adapted CNN with the same layers

In [8]:
class CIFAR10ReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.slope_positive = None
        self.slope_negative = None
        self.do_dropout = False

        self.kernel_size_=(3, 3)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=self.kernel_size_, padding='same')
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=self.kernel_size_, padding='same')
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=self.kernel_size_, padding='same')
        self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=self.kernel_size_, padding='same')
        self.linear5 = nn.Linear(4096, 256)
        self.linear6 = nn.Linear(256, 10)

    def set_slopes(self, slope_positive = 1.0, slope_negative = 0.25):
        self.slope_positive = slope_positive
        self.slope_negative = slope_negative

    def PReLU(self, input: Tensor) -> Tensor:
        input = torch.where(input >= 0, self.slope_positive * input, self.slope_negative * input)
        return input

    def forward_(self, x):
        return self.forward(x)
    
    def forward(self, x):
        #channels*size*size
        #3*32*32->16*32*32 <padding='same'?>
        h1 = self.PReLU(self.conv1(x))
        #16*32*32->32*32*32->32*16*16 <padding='same'?>
        h2 = F.max_pool2d(self.PReLU(self.conv2(h1)), 2)
        if self.do_dropout:
            h2 = F.dropout(h2, 0.25)
        #32*16*16->32*16*16 <padding='same'?>
        h3 = self.PReLU(self.conv3(h2))
        #32*16*16->64*16*16->64*8*8 <padding='same'?>
        h4 = F.max_pool2d(self.PReLU(self.conv4(h3)), 2)
        if self.do_dropout:
            h4 = F.dropout(h4, 0.25)
        #64*8*8->4096->256
        h4flat = torch.flatten(h4, 1)
        h5 = self.PReLU(self.linear5(h4flat))
        if self.do_dropout:
            h5 = F.dropout(h5, 0.5)
        return self.linear6(h5)
    
    def init_weights(self, cb=0.0, cw=1.0):
        logging.debug("CNN weights initialisation with cb={}, cw={}".format(cb, cw))

        #Weight initialisation as in 2.19, 2.20
        self.cb, self.cw = cb, cw
        kernel_size = self.kernel_size_[0]*self.kernel_size_[1]
        self.init_linear_weights(self.conv1, True, cb, cw/kernel_size)
        self.init_linear_weights(self.conv2, True, cb, cw/kernel_size)
        self.init_linear_weights(self.conv3, True, cb, cw/kernel_size)
        self.init_linear_weights(self.conv4, True, cb, cw/kernel_size)
        self.init_linear_weights(self.linear5, True, cb, cw/self.linear5.in_features)
        self.init_linear_weights(self.linear6, True, cb, cw/self.linear6.in_features)

    @staticmethod
    def init_linear_weights(linear, bias_on, var_b=0.0, var_w=1.0):
        nn.init.normal_(linear.weight, mean = 0., std = math.sqrt(var_w)) #approach via torch
        if bias_on:
            nn.init.normal_(linear.bias, mean = 0., std = math.sqrt(var_b))

    @staticmethod
    def init_linear_zeros(linear, bias_on):
        nn.init.zeros_(linear.weight)
        if bias_on:
            nn.init.zeros_(linear.bias)

In [9]:
HISTORY_NORM1 = collections.defaultdict(list)
HISTORY_NORM2 = collections.defaultdict(list)
HISTORY_OPTFIX = collections.defaultdict(list)

In [10]:
def test_loop(testNet, test_loss_meter, test_accuracy_meter):
    for test_batch in test_dataloader:
        images, labels = test_batch
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        with torch.no_grad():
            logits = testNet.forward(images)

            zz_logits = np.transpose(logits.detach().cpu().numpy())
            prediction = logits.argmax(dim=-1).detach()
            loss = loss_crossentropy(zz_logits, labels)
            test_loss_meter.update(loss)
            test_accuracy_meter.update(calculate_accuracy(prediction, labels))

NUM_EPOCH = 50
eta_fixed = 0.01

testOptfix = make_model().to(DEVICE)
loss_fnFix = nn.CrossEntropyLoss()
optimizerFix = optim.SGD(testOptfix.parameters(), lr=eta_fixed, momentum=0.9, nesterov=True)
params = {k: v.detach().cpu().numpy().copy() for k, v in testOptfix.named_parameters()}

stepCalcSoftmax2 = OptimiserEtaSoftmaxArmihoNorm2Base(meta, DEVICE)
testNorm2S = CIFAR10ReLU()
testNorm2S.set_slopes(slope_plus, slope_minus)
testNorm2S.do_dropout = False
testNorm2S.to(DEVICE)
init_weights_copy(testNorm2S, params)

stepCalcSoftmax1 = OptimiserEtaSoftmaxArmihoNorm1Base(meta, DEVICE)
testNorm1S = CIFAR10ReLU()
testNorm1S.set_slopes(slope_plus, slope_minus)
testNorm1S.do_dropout = False
testNorm1S.to(DEVICE)
init_weights_copy(testNorm1S, params)

#qu = collections.deque(10*[1.0], 10)
use_ones=False
for epoch in range(NUM_EPOCH): #eta_fixed if epoch < 5 else eta_fixed * 0.1 if epoch < 10 else eta_fixed * 0.01
    do_dropout = not(epoch < 5)
    use_fix = do_dropout
    use_ones= not(epoch < 1) #not(epoch < 1) # #

    eta_min = 0.01 if epoch < 1 else 0.00001 if not do_dropout else 0.01 if epoch < 20 else 0.001
    eta_max = 0.1 #if epoch < 20 else 0.001
    c1 = 0.001 #if epoch < 1 else 0.01 if epoch < 5 else 0.0125 if epoch < 15 else 0.015
    c2 = 0.999 if epoch < 1 else 0.8 #if epoch < 5 else 0.925 if epoch < 15 else 0.9999

    stepCalcSoftmax1.eta_min = eta_min
    stepCalcSoftmax1.eta_max = eta_max
    stepCalcSoftmax1.stepProcessor.c1 = c1
    stepCalcSoftmax1.stepProcessor.c2 = c2
    testNorm1S.do_dropout = do_dropout

    stepCalcSoftmax2.eta_min = eta_min
    stepCalcSoftmax2.eta_max = eta_max
    stepCalcSoftmax2.stepProcessor.c1 = c1
    stepCalcSoftmax2.stepProcessor.c2 = c2
    testNorm2S.do_dropout = do_dropout
    '''
    c1 = 0.001 if epoch < 1 else 0.01 if epoch < 5 else 0.0125 if epoch < 15 else 0.015 #if epoch < 20 else 0.0175
    c2 = 0.999 if epoch < 1 else 0.75 if epoch < 5 else 0.925  if epoch < 15 else 0.9999
    stepCalcSoftmax2.stepProcessor.c1 = 0.001 if epoch < 1 else 0.01 if epoch < 5 else 0.0125 if epoch < 20 else 0.015
    stepCalcSoftmax2.stepProcessor.c2 = 0.999 if epoch < 1 else 0.75 if epoch < 5 else 0.925   if epoch < 20 else 0.9999
    stepCalcSoftmax2.stepProcessor.c1 = 0.001 if epoch < 1 else 0.01 if epoch < 5 else 0.0125 if epoch < 20 else 0.015
    stepCalcSoftmax2.stepProcessor.c2 = 0.999 if epoch < 1 else 0.75 if epoch < 5 else 0.925  if epoch < 20 else 0.9999    
    "    stepCalcSoftmax2.stepProcessor.c1 = 0.001 if epoch < 1 else 0.01 if epoch < 5 else 0.0125\n",
    "    stepCalcSoftmax2.stepProcessor.c2 = 0.999 if epoch < 1 else 0.75 if epoch < 5 else 0.95\n",    
    '''
         #0.80 if epoch <= 15 else 0.85 if epoch <= 25 else 0.90
    #stepCalcSoftmax2.c1 = 0.001 if epoch < 10 else 0.01 if epoch < 15 else 0.013
        #eta_fixed * math.exp(-0.075*epoch) if epoch < 20 else eta_fixed * 0.1 * math.exp(-0.075*(epoch-20))
     #!-0.075 -0.085 ?0.185 !?-0.175 !-0.15 !-0.115
    #if epoch < 20 else eta_fixed * 0.1 * math.exp(-0.1*(epoch-20))
    optimizerFix.param_groups[0]['lr'] = \
        eta_fixed if epoch < 20 else eta_fixed * 0.1 if epoch < 40 else eta_fixed * 0.01
    '''
    if len(HISTORY_NORM2['test_loss'])> 2 and do_dropout == False \
        and HISTORY_NORM2['test_loss'][-1] >= HISTORY_NORM2['test_loss'][-2]:
        logging.info("##For norm2-optim-simple do_dropout switched to True")
        do_dropout == True
    '''
    testOptfix.train()
    iter=0
    train_eta2_meter, train_armiho2_meter, train_wolf2_meter = AverageMeter(), AverageMeter(), AverageMeter()
    train_eta1_meter, train_armiho1_meter, train_wolf1_meter = AverageMeter(), AverageMeter(), AverageMeter()

    for train_batch in train_dataloader:
        images, labels = train_batch
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        pp = labels_to_softhot(labels, meta.output_dim)
        iter+=1

        logging.info("##Step with norm2-optim-simple")
        logging.info("##\n --==Epoch={}, iter={}; momentum={}, Nesterov={}, eta_min={}==--"\
                     .format(epoch, iter, 0.9, True, stepCalcSoftmax2.eta_min))
        with torch.no_grad():
            logits = testNorm2S.forward_(images)
            logits_detached = logits.detach().cpu().numpy()
            qq = softmax(np.transpose(logits_detached), axis=(0)) + 1e-8
            qq_active = reduce_to_active(qq, pp)
        logging.info("##\n --==Initial qq for ones: min={}, max={}, avg={}==--"\
                        .format(np.min(qq_active), np.max(qq_active), np.average(qq_active)))
        step_result = stepCalcSoftmax2\
            .step(testNorm2S, labels, images, momentum=0.9, Nesterov=True, use_ones=use_ones, use_fix=use_fix)
        train_eta2_meter.update(step_result.eta_value)
        train_armiho2_meter.update(step_result.ck_armiho)
        train_wolf2_meter.update(step_result.ck_wolf)
        '''
        logging.info("##Step with norm1-optim-simple")
        logging.info("##\n --==Epoch={}, iter={}; momentum={}, Nesterov={}, eta_min={}==--"\
                     .format(epoch, iter, 0.9, True, stepCalcSoftmax1.eta_min))
        with torch.no_grad():
            logits1 = testNorm1S.forward_(images)
            logits1_detached = logits1.detach().cpu().numpy()
            qq1 = softmax(np.transpose(logits1_detached), axis=(0)) + 1e-8
            qq1_active = reduce_to_active(qq1, pp)
        logging.info("##\n --==Initial qq for ones: min={}, max={}, avg={}==--"\
                        .format(np.min(qq1_active), np.max(qq1_active), np.average(qq1_active)))
        step1_result = stepCalcSoftmax1\
            .step(testNorm1S, labels, images, momentum=0.9, Nesterov=True, use_ones=use_ones, use_fix=use_fix)
        train_eta1_meter.update(step1_result.eta_value)
        train_armiho1_meter.update(step1_result.ck_armiho)
        train_wolf1_meter.update(step1_result.ck_wolf)        
        '''
        
        logging.info("##Step with Optim-fix, eta={}".format(optimizerFix.param_groups[0]['lr']))
        logitsOptfix = testOptfix.forward(images)
        lossOptfix = loss_fnFix(logitsOptfix, labels)
        optimizerFix.zero_grad()
        lossOptfix.backward()
        optimizerFix.step()


    #HISTORY_NORM1['train_eta'].append(train_eta1_meter.avg)
    #HISTORY_NORM1['train_armiho'].append(train_armiho1_meter.avg)
    #HISTORY_NORM1['train_wolf'].append(train_wolf1_meter.avg)

    HISTORY_NORM2['train_eta'].append(train_eta2_meter.avg)
    HISTORY_NORM2['train_armiho'].append(train_armiho2_meter.avg)
    HISTORY_NORM2['train_wolf'].append(train_wolf2_meter.avg)

    #testNorm1S.do_dropout = False
    #testNorm2S.do_dropout = False
    #testOptfix.eval()
    # testing loop
    #test_loss1S_meter, test_accuracy1S_meter = AverageMeter(), AverageMeter(),
    #test_loop(testNorm1S, test_loss1S_meter, test_accuracy1S_meter)
    #HISTORY_NORM1['test_loss'].append(test_loss1S_meter.avg)
    #HISTORY_NORM1['test_accuracy'].append(test_accuracy1S_meter.avg)

    test_loss2S_meter, test_accuracy2S_meter = AverageMeter(), AverageMeter(),
    test_loop(testNorm2S, test_loss2S_meter, test_accuracy2S_meter)
    HISTORY_NORM2['test_loss'].append(test_loss2S_meter.avg)
    HISTORY_NORM2['test_accuracy'].append(test_accuracy2S_meter.avg) 

    test_lossOptfix_meter, test_accuracyOptfix_meter = AverageMeter(), AverageMeter(),
    test_loop(testOptfix, test_lossOptfix_meter, test_accuracyOptfix_meter)
    HISTORY_OPTFIX['test_loss'].append(test_lossOptfix_meter.avg)
    HISTORY_OPTFIX['test_accuracy'].append(test_accuracyOptfix_meter.avg)
    
    print_comparison(HISTORY_NORM1, HISTORY_NORM2, HISTORY_OPTFIX)
    print("accuracy:\nnorm1={},\nnorm2={}, \naccuracy Nesterov-fix={}"\
          .format(HISTORY_NORM1['test_accuracy'], HISTORY_NORM2['test_accuracy'], HISTORY_OPTFIX['test_accuracy']))


d:\Projects-my\ml\Ml-readinggroup\Roberts_Yaida\chapterInf\common\optimise_v2.py:76: SyntaxWarning: invalid escape sequence '\i'
  '''


KeyboardInterrupt: 

In [11]:
labels

tensor([7, 1, 8, 1, 8, 9, 0, 5, 3, 2, 7, 3, 9, 9, 9, 8], device='cuda:0')

In [ ]:
HISTORY_NORM2['train_armiho']

[0.012211703150975261,
 0.013667262693540126,
 0.013872920266526053,
 0.014053755872527935,
 0.01419477864085536,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [ ]:
HISTORY_NORM2['train_wolf']

[0.6117334722428202,
 0.7266716330468989,
 0.7339522337903834,
 0.7364256334651896,
 0.7363710296155896,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0]

In [ ]:
HISTORY_NORM2['train_eta']

[0.013510246710515445,
 0.011606200299723424,
 0.013508171893841426,
 0.012849777324799146,
 0.011683510882226945,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.009999999999999844,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.0010000000000000007,
 0.001000000

In [ ]:
print("accuracy norm2={}, \naccuracy Nesterov-fix={}"\
        .format(HISTORY_NORM2['test_accuracy'], HISTORY_OPTFIX['test_accuracy']))

accuracy norm2=[0.503781847133758, 0.5833001592356688, 0.6617237261146497, 0.6825238853503185, 0.7259156050955414, 0.7211385350318471, 0.7219347133757962, 0.748109076433121, 0.761046974522293, 0.7658240445859873, 0.7733877388535032, 0.7715963375796179, 0.772093949044586, 0.7811504777070064, 0.7806528662420382, 0.7898089171974523, 0.7849323248407644, 0.7967754777070064, 0.7972730891719745, 0.79796974522293, 0.8158837579617835, 0.8164808917197452, 0.8183718152866242, 0.8207603503184714, 0.8183718152866242, 0.8193670382165605, 0.8219546178343949, 0.8217555732484076, 0.8227507961783439, 0.823546974522293, 0.8251393312101911, 0.8230493630573248, 0.8217555732484076, 0.8241441082802548, 0.8251393312101911, 0.8239450636942676, 0.8252388535031847, 0.8245421974522293, 0.8261345541401274, 0.8229498407643312, 0.8275278662420382, 0.8249402866242038, 0.8252388535031847, 0.8277269108280255, 0.8251393312101911, 0.8253383757961783, 0.8261345541401274, 0.8244426751592356, 0.8266321656050956, 0.824940286

In [ ]:
#HISTORY_NORM2['train_eta']
base = []
for epoch in range(10):
    base.append(eta_fixed) #* math.exp(-0.075*epoch) if epoch < 20 else eta_fixed * 0.1 * math.exp(-0.075*(epoch-20)))
np.asarray(HISTORY_NORM2['train_eta'])/np.asarray(base[0:38])
        

ValueError: operands could not be broadcast together with shapes (50,) (10,) 

In [ ]:
HISTORY_NORM2['train_eta']

[0.015285067952993671,
 0.009277434863285618,
 0.008607079764250725,
 0.007985162187593722,
 0.007408182206817246,
 0.006872892787909793,
 0.006376281516217808,
 0.005915553643668198,
 0.00548811636094018,
 0.005091564206075456,
 0.004723665527410213,
 0.0043823499246494636,
 0.0040656965974060506,
 0.003771923535631524,
 0.0034993774911115305,
 0.0032465246735835394,
 0.00301216009820852,
 0.0027955801143204006,
 0.0025924599922398815,
 0.0024068696398472445,
 0.0013281892892098963,
 0.0012763028431239866,
 0.0012426129780865067,
 0.001187543631967256,
 0.0011694843494133055,
 0.001134341639877675,
 0.0011460235636987642,
 0.0011489829926686775,
 0.0011306902926686018,
 0.0010899206054821582,
 0.0010913725570778124,
 0.001122022868289991,
 0.0010863638790658383,
 0.0010566156514538257,
 0.001061710730254017,
 0.0010463743320390693,
 0.0010467699578634553,
 0.0010209148612334567,
 0.0010296132952730063,
 0.0010336661643026068,
 0.001050916115744024,
 0.001003426687012818,
 0.0010037535

In [ ]:
HISTORY_NORM2['train_eta']

[0.013527505134022249,
 0.009277434863285711,
 0.0086070797642504,
 0.007985162187594035,
 0.007408182206817095,
 0.0068728927879095755,
 0.006376281516217819,
 0.005915886047137713,
 0.005488863030226298,
 0.0050946015422383036,
 0.004733813318898004,
 0.004392553897358442,
 0.004084760229643128,
 0.003802145267919253,
 0.003557414481612458,
 0.003318384061673349,
 0.0030934619129792712,
 0.002910433806084253,
 0.002754594847485997,
 0.002633615220305129,
 0.0021073081397136053,
 0.002067767349029129,
 0.0019929295644926432,
 0.0020184606533114164,
 0.001983161712651638,
 0.0019687265302203725,
 0.0019240437906109659,
 0.0019659193079068324,
 0.0019383042901715164,
 0.0018669031405471469,
 0.0018762375125915126,
 0.0018800636451614134,
 0.0018896789973937979,
 0.00182804641656338,
 0.0018075817823008733,
 0.0017754402462104022,
 0.001785486739718736,
 0.0018074980987763563,
 0.001767527230945539,
 0.0017606376820712944,
 0.0017279195696054928,
 0.0017417798058429906,
 0.00174980697270